# OJ-gen solved.ac difficulty fine-tuning pipeline

이 노트북은 현재 OJ-gen 프로젝트 구조를 기준으로 동작합니다.

전제:
- 학습 GPU: 80GB VRAM
- 추론 GPU: 48GB VRAM
- 데이터 다운로드: https://ftp.kotori9.dev/oj/data.zip
- 기본 목적: BOJ 문제 본문을 보고 solved.ac 난이도 tier value 1-30을 예측하는 SFT 모델 학습

Unrated, 즉 level 0 문제는 기본적으로 제외합니다. 필요하면 CONFIG에서 INCLUDE_UNRATED를 True로 바꾸면 됩니다.


## 0. Runtime check

In [ ]:
!nvidia-smi

## 1. Project setup

In [ ]:
from pathlib import Path
import os

PROJECT_DIR = Path.cwd()
print(PROJECT_DIR)
print(sorted([p.name for p in PROJECT_DIR.iterdir()])[:50])

In [ ]:
%%bash
set -e
rm -rf .config sample_data
if [ ! -d .git ]; then
  git clone https://github.com/Origin-Glass/OJ-gen.git .
else
  git pull --ff-only || true
fi

In [ ]:
%pip install -e .
%pip install -r requirements-train.txt
%pip install -U datasets scikit-learn pandas

## 2. Configuration

80GB VRAM 학습 기준 기본값입니다. 48GB VRAM에서 추론만 할 경우에는 아래쪽 inference cell을 사용합니다.


In [ ]:
from pathlib import Path

DATA_URL = "https://ftp.kotori9.dev/oj/data.zip"
DATA_ZIP = Path("data.zip")
DATA_DIR = Path("data")
SFT_DIR = DATA_DIR / "sft"
OUT_DIR = Path("outputs")

MODEL_ID = "Qwen/Qwen3.6-35B-A3B"
BASE_OUT = OUT_DIR / "qwen3_6_35b_a3b_solvedac_difficulty_v1"
MAX_SEQ_LENGTH = 8192
LOAD_IN_4BIT = True
INCLUDE_UNRATED = False
VALID_RATIO = 0.03
SEED = 3407

TRAIN_EPOCHS = 1
TRAIN_BATCH_SIZE = 1
GRAD_ACCUM = 8
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.0
LEARNING_RATE = 2e-4

TARGET_TOTAL = 50000
MIN_LOCAL_SCORE = 0.90
MIN_LLM_SCORE = 0.85
MIN_CONFIDENCE = 0.75

SFT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("model", MODEL_ID)
print("output", BASE_OUT)

## 3. Download and extract data

In [ ]:
%%bash
set -e
mkdir -p data
if [ ! -f data.zip ]; then
  if command -v curl >/dev/null 2>&1; then
    curl -L --retry 5 --retry-delay 5 -o data.zip https://ftp.kotori9.dev/oj/data.zip
  else
    wget -O data.zip https://ftp.kotori9.dev/oj/data.zip
  fi
fi
python - <<'PYINNER'
import zipfile
from pathlib import Path
z = Path('data.zip')
with zipfile.ZipFile(z) as f:
    f.extractall('data')
print('extracted', z, 'to data')
PYINNER
find data -maxdepth 3 -type f | sort | head -80

## 4. Locate dataset files

In [ ]:
from pathlib import Path

candidates = []
for name in ["boj_difficulty_classification.jsonl", "boj_sft_messages.jsonl", "boj_problems_indexed.jsonl"]:
    candidates.extend(DATA_DIR.rglob(name))

for p in candidates:
    print(p)

classification_path = next((p for p in DATA_DIR.rglob("boj_difficulty_classification.jsonl")), None)
sft_messages_path = next((p for p in DATA_DIR.rglob("boj_sft_messages.jsonl")), None)
problems_path = next((p for p in DATA_DIR.rglob("boj_problems_indexed.jsonl")), None)

print("classification_path", classification_path)
print("sft_messages_path", sft_messages_path)
print("problems_path", problems_path)

assert classification_path is not None or sft_messages_path is not None or problems_path is not None

## 5. Build SFT dataset

우선순위:
1. boj_difficulty_classification.jsonl에서 일관된 SFT 메시지 재생성
2. 없으면 boj_sft_messages.jsonl 사용
3. 없으면 boj_problems_indexed.jsonl에서 재생성


In [ ]:
import json
from collections import Counter
from pathlib import Path

TIER_NAMES = {
    0: "Unrated",
    1: "Bronze V", 2: "Bronze IV", 3: "Bronze III", 4: "Bronze II", 5: "Bronze I",
    6: "Silver V", 7: "Silver IV", 8: "Silver III", 9: "Silver II", 10: "Silver I",
    11: "Gold V", 12: "Gold IV", 13: "Gold III", 14: "Gold II", 15: "Gold I",
    16: "Platinum V", 17: "Platinum IV", 18: "Platinum III", 19: "Platinum II", 20: "Platinum I",
    21: "Diamond V", 22: "Diamond IV", 23: "Diamond III", 24: "Diamond II", 25: "Diamond I",
    26: "Ruby V", 27: "Ruby IV", 28: "Ruby III", 29: "Ruby II", 30: "Ruby I",
}

SYSTEM_PROMPT = "You are a BOJ problem difficulty estimator. Predict the solved.ac tier value from 1 to 30 from the problem statement. Return only the requested fields."


def clean_text(x):
    if x is None:
        return ""
    return str(x).replace("\r\n", "\n").replace("\r", "\n").strip()


def make_user_text(obj):
    pid = obj.get("problem_id") or obj.get("problemId") or obj.get("id") or ""
    title = clean_text(obj.get("title"))
    desc = clean_text(obj.get("description"))
    inp = clean_text(obj.get("input_description"))
    out = clean_text(obj.get("output_description"))
    samples = obj.get("samples") or []
    parts = []
    parts.append(f"Problem ID: {pid}")
    if title:
        parts.append(f"Title: {title}")
    if desc:
        parts.append("Description:\n" + desc)
    if inp:
        parts.append("Input:\n" + inp)
    if out:
        parts.append("Output:\n" + out)
    if samples:
        sample_parts = []
        for i, s in enumerate(samples[:3], 1):
            si = clean_text(s.get("input") if isinstance(s, dict) else "")
            so = clean_text(s.get("output") if isinstance(s, dict) else "")
            sample_parts.append(f"Sample {i} Input:\n{si}\nSample {i} Output:\n{so}")
        parts.append("Samples:\n" + "\n\n".join(sample_parts))
    parts.append("Predict solved.ac difficulty. Use tier_value 1-30. Also provide tier_name and tier_group.")
    return "\n\n".join(parts)


def tier_group(level):
    if level <= 0:
        return "unrated"
    groups = ["bronze", "silver", "gold", "platinum", "diamond", "ruby"]
    return groups[(level - 1) // 5]


def make_assistant_text(level):
    level = int(level)
    return f"tier_value: {level}\ntier_name: {TIER_NAMES.get(level, 'Unknown')}\ntier_group: {tier_group(level)}"


def read_jsonl(path):
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)

records = []
source_name = None

if classification_path is not None:
    source_name = str(classification_path)
    for obj in read_jsonl(classification_path):
        level = obj.get("label") or obj.get("difficulty_index") or obj.get("level")
        if level is None:
            continue
        level = int(level)
        if level == 0 and not INCLUDE_UNRATED:
            continue
        if "text" in obj and not obj.get("description"):
            user_text = clean_text(obj["text"]) + "\n\nPredict solved.ac difficulty. Use tier_value 1-30. Also provide tier_name and tier_group."
        else:
            user_text = make_user_text(obj)
        records.append({
            "problem_id": obj.get("problem_id") or obj.get("problemId"),
            "level": level,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_text},
                {"role": "assistant", "content": make_assistant_text(level)},
            ],
        })
elif sft_messages_path is not None:
    source_name = str(sft_messages_path)
    for obj in read_jsonl(sft_messages_path):
        msgs = obj.get("messages")
        if not msgs:
            continue
        assistant = msgs[-1].get("content", "") if isinstance(msgs[-1], dict) else ""
        level = None
        for line in assistant.splitlines():
            if "difficulty_index" in line or "tier_value" in line:
                digits = "".join(ch for ch in line if ch.isdigit())
                if digits:
                    level = int(digits)
                    break
        if level == 0 and not INCLUDE_UNRATED:
            continue
        records.append({"problem_id": obj.get("problem_id"), "level": level, "messages": msgs})
else:
    source_name = str(problems_path)
    for obj in read_jsonl(problems_path):
        level = obj.get("difficulty_index") or obj.get("level")
        if level is None:
            solved = obj.get("solvedac") or obj.get("solved_ac") or {}
            level = solved.get("level") if isinstance(solved, dict) else None
        if level is None:
            continue
        level = int(level)
        if level == 0 and not INCLUDE_UNRATED:
            continue
        records.append({
            "problem_id": obj.get("problem_id") or obj.get("problemId"),
            "level": level,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": make_user_text(obj)},
                {"role": "assistant", "content": make_assistant_text(level)},
            ],
        })

print("source", source_name)
print("records", len(records))
print("level distribution", Counter(r["level"] for r in records).most_common())
assert records

## 6. Train and validation split

In [ ]:
from sklearn.model_selection import train_test_split
from collections import Counter
import json

levels = [r["level"] for r in records]
train_records, valid_records = train_test_split(
    records,
    test_size=VALID_RATIO,
    random_state=SEED,
    stratify=levels,
)

train_path = SFT_DIR / "solvedac_difficulty_train.jsonl"
valid_path = SFT_DIR / "solvedac_difficulty_valid.jsonl"
full_path = SFT_DIR / "solvedac_difficulty_full.jsonl"

for path, rows in [(train_path, train_records), (valid_path, valid_records), (full_path, records)]:
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps({"messages": r["messages"], "problem_id": r["problem_id"], "level": r["level"]}, ensure_ascii=False) + "\n")
    print(path, len(rows))

print("train", Counter(r["level"] for r in train_records).most_common())
print("valid", Counter(r["level"] for r in valid_records).most_common())

## 7. Quick dataset inspection

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files=str(train_path), split="train")
print(ds)
print(ds[0]["messages"][0])
print(ds[0]["messages"][-1])

## 8. Primary training on 80GB VRAM

OJ-gen의 train_sft 모듈을 그대로 사용합니다. 80GB VRAM 기준으로 QLoRA 35B급 모델을 목표로 합니다.


In [ ]:
!CUDA_VISIBLE_DEVICES=0 python -m ojgen.train_sft \
  --data {train_path} \
  --out {BASE_OUT} \
  --model {MODEL_ID} \
  --max-seq-length {MAX_SEQ_LENGTH} \
  --epochs {TRAIN_EPOCHS} \
  --batch-size {TRAIN_BATCH_SIZE} \
  --grad-accum {GRAD_ACCUM} \
  --lora-r {LORA_R} \
  --lora-alpha {LORA_ALPHA} \
  --lora-dropout {LORA_DROPOUT} \
  --target-modules qv \
  --optim paged_adamw_8bit \
  --dataset-num-proc 1 \
  --no-packing

## 9. Optional smaller smoke training

큰 모델 학습 전에 파이프라인만 검증할 때 사용합니다.


In [ ]:
SMOKE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
SMOKE_OUT = OUT_DIR / "smoke_qwen2_5_7b_solvedac_difficulty"
SMOKE_MAX_SAMPLES = 512

smoke_path = SFT_DIR / "solvedac_difficulty_smoke.jsonl"
with smoke_path.open("w", encoding="utf-8") as f:
    for r in train_records[:SMOKE_MAX_SAMPLES]:
        f.write(json.dumps({"messages": r["messages"], "problem_id": r["problem_id"], "level": r["level"]}, ensure_ascii=False) + "\n")
print(smoke_path, SMOKE_MAX_SAMPLES)

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python -m ojgen.train_sft \
  --data {smoke_path} \
  --out {SMOKE_OUT} \
  --model {SMOKE_MODEL_ID} \
  --max-seq-length 4096 \
  --epochs 1 \
  --batch-size 1 \
  --grad-accum 4 \
  --lora-r 8 \
  --lora-alpha 16 \
  --lora-dropout 0.0 \
  --target-modules qv \
  --optim paged_adamw_8bit \
  --dataset-num-proc 1 \
  --no-packing

## 10. Validation sample generation on 48GB VRAM

48GB VRAM 추론 환경에서는 base model 4bit 로드 + LoRA adapter 방식으로 실행하는 것을 권장합니다. 아래 cell은 adapter가 저장된 뒤 검증 샘플 몇 개를 생성합니다.


In [ ]:
import json
from pathlib import Path

valid_preview_path = SFT_DIR / "valid_preview_16.jsonl"
with valid_preview_path.open("w", encoding="utf-8") as f:
    for r in valid_records[:16]:
        user_msgs = [m for m in r["messages"] if m["role"] != "assistant"]
        f.write(json.dumps({"messages": user_msgs, "expected_level": r["level"], "problem_id": r["problem_id"]}, ensure_ascii=False) + "\n")
print(valid_preview_path)

In [ ]:
from unsloth import FastLanguageModel
import torch
import json

adapter_path = str(BASE_OUT)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=adapter_path,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

sample = valid_records[0]
prompt_messages = [m for m in sample["messages"] if m["role"] != "assistant"]
text = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors="pt").to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens=64,
    temperature=0.0,
    do_sample=False,
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
print("expected", sample["level"], TIER_NAMES.get(sample["level"]))

## 11. Simple validation metric

생성 결과에서 tier_value를 정규식으로 추출해 exact accuracy와 mean absolute error를 계산합니다.


In [ ]:
import re
import torch


def extract_level(text):
    m = re.search(r"tier[_ ]?value\s*:\s*(\d+)", text, re.IGNORECASE)
    if not m:
        m = re.search(r"difficulty[_ ]?index\s*:\s*(\d+)", text, re.IGNORECASE)
    if not m:
        return None
    v = int(m.group(1))
    if 0 <= v <= 30:
        return v
    return None

N = min(100, len(valid_records))
correct = 0
abs_errors = []
missing = 0

for sample in valid_records[:N]:
    prompt_messages = [m for m in sample["messages"] if m["role"] != "assistant"]
    text = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to("cuda")
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=64, temperature=0.0, do_sample=False)
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    pred = extract_level(decoded)
    gold = int(sample["level"])
    if pred is None:
        missing += 1
        continue
    correct += int(pred == gold)
    abs_errors.append(abs(pred - gold))

print("n", N)
print("parsed", len(abs_errors))
print("missing", missing)
print("exact_accuracy", correct / max(1, N))
print("mae", sum(abs_errors) / max(1, len(abs_errors)))

## 12. Save run metadata

In [ ]:
import json
from datetime import datetime

metadata = {
    "model_id": MODEL_ID,
    "output_dir": str(BASE_OUT),
    "data_url": DATA_URL,
    "source_name": source_name,
    "max_seq_length": MAX_SEQ_LENGTH,
    "include_unrated": INCLUDE_UNRATED,
    "train_size": len(train_records),
    "valid_size": len(valid_records),
    "seed": SEED,
    "train_gpu_vram_gb": 80,
    "inference_gpu_vram_gb": 48,
    "created_at": datetime.utcnow().isoformat() + "Z",
}
meta_path = BASE_OUT / "run_metadata.json"
meta_path.parent.mkdir(parents=True, exist_ok=True)
meta_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
print(meta_path)
print(json.dumps(metadata, ensure_ascii=False, indent=2))

## 13. Optional packaging

In [ ]:
%%bash
set -e
if [ -d outputs/qwen3_6_35b_a3b_solvedac_difficulty_v1 ]; then
  tar -czf outputs/qwen3_6_35b_a3b_solvedac_difficulty_v1_adapter.tar.gz outputs/qwen3_6_35b_a3b_solvedac_difficulty_v1
  ls -lh outputs/qwen3_6_35b_a3b_solvedac_difficulty_v1_adapter.tar.gz
fi